In [ ]:
import pandas as pd
from pathlib import Path

BASE        = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
ARQUIVO_IN  = BASE / "data/raw/ibge/Tab_Compl_CNT_4T25.xls"
ARQUIVO_OUT = BASE / "data/processed/pib_setorial_2012_2023.csv"

In [ ]:
df_raw = pd.read_excel(ARQUIVO_IN, sheet_name="Valores Correntes", header=None)

print(f"Shape bruto: {df_raw.shape}")

colunas_map = {
    0:  "periodo",
    1:  "agropecuaria",
    2:  "ind_extrativa",
    3:  "ind_transformacao",
    4:  "eletricidade_agua",
    5:  "construcao",
    7:  "comercio",
    8:  "transporte",
    9:  "info_comunicacao",
    10: "financeiro",
    11: "imobiliario",
    12: "outros_servicos",
    13: "adm_publica_saude_educ",
    17: "pib_total",
}

df = df_raw.iloc[4:, list(colunas_map.keys())].copy()
df.columns = list(colunas_map.values())

In [ ]:
df["periodo"] = df["periodo"].astype(str).str.strip()

mask_anual = df["periodo"].str.match(r"^\d{4}$")
df_anual   = df[mask_anual].copy()

df_anual["ano"] = df_anual["periodo"].astype(int)

df_anual = df_anual[(df_anual["ano"] >= 2012) & (df_anual["ano"] <= 2023)].copy()

for col in df_anual.columns:
    if col not in ["periodo", "ano"]:
        df_anual[col] = pd.to_numeric(df_anual[col], errors="coerce")

df_anual = df_anual.drop(columns=["periodo"]).reset_index(drop=True)

print(f"\nAnos disponíveis: {sorted(df_anual['ano'].tolist())}")
print(f"Shape final: {df_anual.shape}")

In [ ]:
print("\n── Valores Adicionados por atividade (R$ milhões correntes) ──")
print(df_anual.set_index("ano").to_string())

In [ ]:
df_anual.to_csv(ARQUIVO_OUT, index=False)
print(f"\n✓ Salvo em: {ARQUIVO_OUT}")
print(f"  Tamanho: {ARQUIVO_OUT.stat().st_size / 1024:.1f} KB")